# NarraBERT event relation model

[`CLS-Lab/narrative-event-relation-roberta`](https://huggingface.co/CLS-Lab/narrative-event-relation-roberta)
takes a passage with two event triggers marked, and returns two logits: **temporal** (are the events
sequenced?) and **causal** (did the first cause or enable the second?).

The initial event triggers are detected from the LitBank event detector.

In [4]:
import json

import pandas as pd
import torch
from torch import nn
from datasets import load_dataset
from huggingface_hub import hf_hub_download
from sklearn.metrics import f1_score
from transformers import AutoModel, AutoTokenizer

REPO_ID = "CLS-Lab/narrative-event-relation-roberta"
ENTITY_MARKERS = ["[E1]", "[/E1]", "[E2]", "[/E2]"]

device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)


class EventRelationRoBERTa(nn.Module):
    """RoBERTa backbone + a temporal and a causal binary head, on [CLS]."""

    def __init__(self, model_name, n_new_tokens):
        super().__init__()
        self.backbone = AutoModel.from_pretrained(model_name)
        # Mandatory: training added the 4 markers to the vocab (50265 -> 50269).
        self.backbone.resize_token_embeddings(self.backbone.config.vocab_size + n_new_tokens)
        hidden = self.backbone.config.hidden_size
        self.temporal_head = nn.Linear(hidden, 1)
        self.causal_head = nn.Linear(hidden, 1)

    def forward(self, input_ids, attention_mask):
        cls = self.backbone(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state[:, 0]
        return self.temporal_head(cls).squeeze(-1), self.causal_head(cls).squeeze(-1)


# The checkpoint is a raw state_dict, so AutoModel.from_pretrained(REPO_ID) will not work.
config = json.load(open(hf_hub_download(REPO_ID, "config.json")))
MAX_LEN = config["max_len"]

tokenizer = AutoTokenizer.from_pretrained(REPO_ID, subfolder="tokenizer")  # already knows the markers
model = EventRelationRoBERTa(config["model_name"], len(ENTITY_MARKERS))
# Load on CPU first; loading straight to mps raises "Unaligned blit request".
model.load_state_dict(
    torch.load(hf_hub_download(REPO_ID, "model.pt"), map_location="cpu", weights_only=True)
)
model.to(device).eval()

print(f"{device}, max_len={MAX_LEN}, vocab={len(tokenizer)}")

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-base and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


mps, max_len=256, vocab=50269


## Input format

A span is `[char_start, char_end, token]` with `text[start:end] == token`. The two spans are wrapped in
literal `[E1]`/`[E2]` markers before tokenizing.

Any span source works — spaCy verbs, an LLM, hand annotation. The paper used the LitBank event detector
(Sims et al., 2019), which is not included here.

In [5]:
def insert_markers(text, span1, span2):
    """Wrap two spans in [E1]/[E2] markers. Inserts right-to-left so offsets stay valid."""
    insertions = sorted(
        [(span1[0], "[E1]"), (span1[1], "[/E1]"), (span2[0], "[E2]"), (span2[1], "[/E2]")],
        key=lambda x: -x[0],
    )
    for pos, marker in insertions:
        text = text[:pos] + marker + text[pos:]
    return text


@torch.no_grad()
def predict(pairs, batch_size=32):
    """pairs: list of (text, span1, span2). Returns a DataFrame of logits and binary labels."""
    marked = [insert_markers(*p) for p in pairs]
    t_out, c_out = [], []
    for i in range(0, len(marked), batch_size):
        enc = tokenizer(
            marked[i:i + batch_size],
            max_length=MAX_LEN, padding="max_length", truncation=True, return_tensors="pt",
        )
        t, c = model(enc["input_ids"].to(device), enc["attention_mask"].to(device))
        t_out.extend(t.reshape(-1).cpu().float().tolist())  # reshape guards the batch-of-1 case
        c_out.extend(c.reshape(-1).cpu().float().tolist())
    return pd.DataFrame({
        "temporal_logit": t_out, "temporal_sequential": [x > 0 for x in t_out],
        "causal_logit": c_out, "causal": [x > 0 for x in c_out],
    })


text = "She opened the door and the cat escaped."
print(insert_markers(text, [4, 10, "opened"], [32, 39, "escaped"]))
predict([(text, [4, 10, "opened"], [32, 39, "escaped"])]).round(3)

She [E1]opened[/E1] the door and the cat [E2]escaped[/E2].


,temporal_logit,temporal_sequential,causal_logit,causal
0,5.012,True,4.452,True


## Reading the heads

Both heads are binary collapses of a richer annotation scheme:

| Head | `True` | `False` |
|---|---|---|
| `temporal_sequential` | `span1_first`, `span2_first` | `simultaneous`, `same_event`, `too_hard_to_tell` |
| `causal` | `direct_cause`, `enables` | `not_related` |

So `temporal_sequential=True` means the events are **ordered**, not that span1 came first. And `causal=False`
merges "unrelated" with "too hard to tell", so a logit near 0 reads as uncertainty, not confident independence.

## Evaluating against human gold labels

[`CLS-Lab/narrative-gold-annotations`](https://huggingface.co/datasets/CLS-Lab/narrative-gold-annotations)
`event_relation` holds 440 gold event pairs. Pairs where either span is not a genuine event trigger have
null relation labels, so each head is scored on its own applicable subset.

In [6]:
gold = load_dataset("CLS-Lab/narrative-gold-annotations", "event_relation", split="train").to_pandas()

pred = predict([
    (r.sampled_text, json.loads(r.assigned_span1), json.loads(r.assigned_span2))
    for r in gold.itertuples()
])

# Binarize the gold labels the same way training did. Null = the pair was not applicable
# (one of the spans is not a genuine event trigger), so each head is scored on its own subset.
HEADS = {
    "temporal_sequential": ("temporal_order_gold", ["span1_first", "span2_first"]),
    "causal": ("causality_rating_gold", ["direct_cause", "enables"]),
}

rows = []
for head, (col, positive) in HEADS.items():
    keep = gold[col].notna()
    y_true, y_pred = gold.loc[keep, col].isin(positive), pred.loc[keep, head]
    rows.append({
        "n": int(keep.sum()),
        "positive_rate": y_true.mean(),
        "predicted_rate": y_pred.mean(),
        "accuracy": (y_pred == y_true).mean(),
        "macro_f1": f1_score(y_true, y_pred, average="macro"),
    })

pd.DataFrame(rows, index=list(HEADS)).round(3)

,n,positive_rate,predicted_rate,accuracy,macro_f1
temporal_sequential,236,0.818,0.788,0.852,0.766
causal,203,0.547,0.532,0.729,0.727
